# CYS5550: Introduction to Artificial Intelligence
## Miniproject 2 — Movie Review Sentiment: From Word Counts to Language Models

In this miniproject you will build a sentiment classifier for movie reviews and compare four levels of text representation:

| Part | What you build | Model family |
|---|---|---|
| 0 | Set-up | — |
| 1 | Data and preprocessing | — |
| 2 | Word counts and TF-IDF features | Naive Bayes, Logistic Regression, SVM |
| 3 | Word embeddings | Word2Vec and BERT, with Logistic Regression and MLP |
| 4 | A small local language model | TinyLlama (prompting) |

**Dataset:** NLTK `movie_reviews`, 2,000 IMDb reviews (1,000 positive, 1,000 negative). Every method is evaluated on the same 400 held-out test reviews.

**Submit:** this notebook with all cells run and outputs visible.

## Part 0 — Set-up (VS Code)

**Requirements:** Python 3.10–3.12, about 8 GB RAM (16 GB is more comfortable), about 3 GB free disk.

**Download sizes (first run only):** NLTK movie reviews ~3 MB · `bert-base-uncased` ~440 MB · TinyLlama chat model ~2.2 GB.

1. Install **Python** from [python.org](https://www.python.org/downloads/). On Windows, tick *Add Python to PATH*.
2. Install **VS Code**, then add the **Python** and **Jupyter** extensions (Microsoft).
3. Open this notebook's folder in VS Code (**File → Open Folder**).
4. Open a terminal (**Terminal → New Terminal**) and run:

```bash
python -m venv .venv
# macOS / Linux:
source .venv/bin/activate
# Windows PowerShell:
.\.venv\Scripts\Activate.ps1

pip install numpy pandas scikit-learn matplotlib nltk gensim transformers torch accelerate ipykernel
```

5. Open the notebook, click **Select Kernel → Python Environments → `.venv`** (top right).
6. Run the set-up cell below. Use **Run All** from the toolbar to run everything.

> If something fails with an import error: fix the install in the terminal, then click **Restart** (↻) in the notebook toolbar and run again.

In [18]:
import nltk
nltk.download("movie_reviews", quiet=True)
from nltk.corpus import movie_reviews
print("Reviews available:", len(movie_reviews.fileids()))

Reviews available: 2000


## Part 1 — Data and Preprocessing

Load the reviews, then split them into training (80%) and test (20%) sets. The split is stratified, so both sets have equal numbers of positive and negative reviews.

In [19]:
import re
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from nltk.corpus import movie_reviews

fileids = movie_reviews.fileids()
raw_docs = [movie_reviews.raw(f) for f in fileids]
labels   = [f.split("/")[0] for f in fileids]          # 'pos' or 'neg'

# Split the RAW text first. Part 3.2 (BERT) and Part 4 (TinyLlama) use the raw text.
raw_train, raw_test, y_train, y_test = train_test_split(
    raw_docs, labels, test_size=0.2, random_state=42, stratify=labels)

print("Train:", len(raw_train), " Test:", len(raw_test))
print("Example (raw):", raw_train[0][:200].replace("\n", " "), "...")

Train: 1600  Test: 400
Example (raw): saving private ryan ( dreamworks ) running time : 2 hours 48 minutes .  starring tom hanks , edward burns , tom sizemore and matt damon directed by steven spielberg already being hailed as the 'greate ...


In [20]:
def clean_text(text):
    """Lowercase, turn "n't" into " not", keep only letters, return tokens joined by spaces."""
    text = text.lower().replace("n't", " not")
    tokens = re.findall(r"[a-z]+", text)
    return " ".join(tokens)

print(clean_text("This movie isn't GOOD, but the actors were 100% brilliant!"))

this movie is not good but the actors were brilliant


In [21]:
X_train = [clean_text(t) for t in raw_train]
X_test  = [clean_text(t) for t in raw_test]
print("Cleaned example:", X_train[0][:150], "...")

Cleaned example: saving private ryan dreamworks running time hours minutes starring tom hanks edward burns tom sizemore and matt damon directed by steven spielberg alr ...


## Part 2 — Classical Machine Learning

We turn each review into a vector of numbers, then train a standard classifier. We use scikit-learn directly, one step at a time (no `Pipeline`), so every step is visible.

**A note on "linear regression":** for a yes/no label, we use **Logistic Regression**. It is the linear model used for classification: it fits a weighted sum of word features and turns it into a probability of "positive".

`results` collects the accuracy of each model so we can compare them at the end.

In [22]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

results = {}   # model name -> test accuracy

### 2.1 Baseline: Bag of Words + Naive Bayes

Bag of words counts how often each word appears. Naive Bayes uses those counts to estimate how likely each word is under "positive" and "negative".

In [23]:
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)     # learn the vocabulary from TRAIN only
X_test_bow  = bow.transform(X_test)

nb = MultinomialNB()
nb.fit(X_train_bow, y_train)

acc = accuracy_score(y_test, nb.predict(X_test_bow))
results["BoW + Naive Bayes"] = acc
print(f"Vocabulary size: {len(bow.vocabulary_):,}")
print(f"BoW + Naive Bayes accuracy: {acc:.3f}")

Vocabulary size: 35,686
BoW + Naive Bayes accuracy: 0.815


### 2.2 Baseline: TF-IDF + Logistic Regression

TF-IDF is a bag-of-words that down-weights words appearing in almost every review (like *film* or *movie*) and up-weights words that are distinctive.

In [24]:
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_tfidf, y_train)

acc = accuracy_score(y_test, logreg.predict(X_test_tfidf))
results["TF-IDF + Logistic Regression"] = acc
print(f"TF-IDF + Logistic Regression accuracy: {acc:.3f}")

TF-IDF + Logistic Regression accuracy: 0.820


### 2.3 Different features + SVM

A Support Vector Machine (`LinearSVC`) finds the line that separates the two classes with the widest margin. Here we compare two feature sets with the same SVM:

- **unigrams:** single words only (`ngram_range=(1, 1)`)
- **uni + bigrams:** single words and two-word phrases such as *not good* or *waste of* (`ngram_range=(1, 2)`)

Phrases matter in sentiment: *not good* means something different from *good*.

In [25]:
# Unigram features + SVM
tfidf_uni = TfidfVectorizer(ngram_range=(1, 1), min_df=2)
svm_uni = LinearSVC(C=1.0).fit(tfidf_uni.fit_transform(X_train), y_train)
acc_uni = accuracy_score(y_test, svm_uni.predict(tfidf_uni.transform(X_test)))
results["TF-IDF unigrams + SVM"] = acc_uni

# Uni + bigram features + SVM
tfidf_bi = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_train_bi = tfidf_bi.fit_transform(X_train)
svm_bi = LinearSVC(C=1.0).fit(X_train_bi, y_train)
acc_bi = accuracy_score(y_test, svm_bi.predict(tfidf_bi.transform(X_test)))
results["TF-IDF uni+bigrams + SVM"] = acc_bi

print(f"TF-IDF unigrams + SVM:    {acc_uni:.3f}  (features: {len(tfidf_uni.vocabulary_):,})")
print(f"TF-IDF uni+bigrams + SVM: {acc_bi:.3f}  (features: {len(tfidf_bi.vocabulary_):,})")

# Which phrases push a review toward "positive" or "negative"?
names = tfidf_bi.get_feature_names_out()
coef = svm_bi.coef_[0]              # positive weight -> "pos"
order = np.argsort(coef)
print("\nMost negative features:", [names[i] for i in order[:10]])
print("Most positive features:", [names[i] for i in order[-10:][::-1]])

TF-IDF unigrams + SVM:    0.853  (features: 21,474)
TF-IDF uni+bigrams + SVM: 0.858  (features: 115,477)

Most negative features: ['bad', 'worst', 'only', 'plot', 'nothing', 'boring', 'have', 'no', 'harry', 'script']
Most positive features: ['and', 'great', 'as', 'well', 'very', 'is', 'also', 'life', 'the best', 'war']


## Part 3 — Word Embeddings

An **embedding** represents each word as a dense vector of numbers, where words with similar meanings get similar vectors. To represent a whole review, we average the vectors of its words. The classifier then learns from that average.

### 3.1 Word2Vec (trained on our reviews)

We train a small Word2Vec model on the **training reviews only**, so the test reviews never influence the embeddings. The model learns from word co-occurrence: words that appear near each other get similar vectors.

In [26]:
from gensim.models import Word2Vec

train_sentences = [t.split() for t in X_train]   # list of token lists
w2v = Word2Vec(train_sentences, vector_size=100, window=5, min_count=2,
               epochs=20, workers=4, seed=42)

print("Vocabulary:", len(w2v.wv), "words")
print("Nearest neighbours of 'good':", [w for w, _ in w2v.wv.most_similar("good", topn=5)])
print("Nearest neighbours of 'boring':", [w for w, _ in w2v.wv.most_similar("boring", topn=5)])

Vocabulary: 22941 words
Nearest neighbours of 'good': ['great', 'decent', 'bad', 'nice', 'funny']
Nearest neighbours of 'boring': ['predictable', 'unfunny', 'pointless', 'silly', 'contrived']


In [27]:
def doc_vector(text, model):
    """Average the Word2Vec vectors of all known words in the review."""
    vectors = [model.wv[w] for w in text.split() if w in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

In [28]:
X_train_w2v = np.vstack([doc_vector(t, w2v) for t in X_train])
X_test_w2v  = np.vstack([doc_vector(t, w2v) for t in X_test])
print("Feature matrix:", X_train_w2v.shape)

Feature matrix: (1600, 100)


**3.1a — Word2Vec + Logistic Regression** (the linear model)

In [29]:
w2v_lr = LogisticRegression(max_iter=1000).fit(X_train_w2v, y_train)
acc = accuracy_score(y_test, w2v_lr.predict(X_test_w2v))
results["Word2Vec + Logistic Regression"] = acc
print(f"Word2Vec + Logistic Regression accuracy: {acc:.3f}")

Word2Vec + Logistic Regression accuracy: 0.797


**3.1b — Word2Vec + MLP** (a small neural network with one hidden layer of 64 units)

In [30]:
from sklearn.neural_network import MLPClassifier

w2v_mlp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42)
w2v_mlp.fit(X_train_w2v, y_train)
acc = accuracy_score(y_test, w2v_mlp.predict(X_test_w2v))
results["Word2Vec + MLP"] = acc
print(f"Word2Vec + MLP accuracy: {acc:.3f}")

Word2Vec + MLP accuracy: 0.772


c:\Users\hanna\source\repos\MiniProjectDeesStacy\.venv313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


### 3.2 BERT embeddings

BERT is a pre-trained Transformer. Instead of training our own embeddings, we use BERT's output vectors for each review. The first run downloads `bert-base-uncased` (~440 MB). We cut each review to its first 256 tokens to keep this fast on a laptop; this takes roughly 5–15 minutes on a CPU.

To represent a review, we average BERT's token vectors, ignoring padding.

In [31]:
import torch
from transformers import AutoTokenizer, AutoModel

BERT_ID = "bert-base-uncased"
bert_tok = AutoTokenizer.from_pretrained(BERT_ID)
bert = AutoModel.from_pretrained(BERT_ID).eval()
print("BERT loaded.")

c:\Users\hanna\source\repos\MiniProjectDeesStacy\.venv313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 488.34it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ign

BERT loaded.


In [32]:
@torch.no_grad()
def bert_embed(texts, batch_size=16, max_len=256):
    """Return one 768-dim vector per review: the mean of BERT's token vectors."""
    all_vecs = []
    for start in range(0, len(texts), batch_size):
        batch = bert_tok(texts[start:start + batch_size], padding=True,
                         truncation=True, max_length=max_len, return_tensors="pt")
        hidden = bert(**batch).last_hidden_state              # (batch, tokens, 768)
        mask = batch["attention_mask"].unsqueeze(-1).float()  # 1 for real tokens, 0 for padding
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
        all_vecs.append(pooled.numpy())
    return np.vstack(all_vecs)

In [33]:
# Use the RAW reviews: BERT's tokenizer handles capitalisation and punctuation itself.
X_train_bert = bert_embed(raw_train)
X_test_bert  = bert_embed(raw_test)
print("BERT feature matrix:", X_train_bert.shape)

BERT feature matrix: (1600, 768)


**3.2a — BERT + Logistic Regression**

In [34]:
bert_lr = LogisticRegression(max_iter=1000).fit(X_train_bert, y_train)
acc = accuracy_score(y_test, bert_lr.predict(X_test_bert))
results["BERT + Logistic Regression"] = acc
print(f"BERT + Logistic Regression accuracy: {acc:.3f}")

BERT + Logistic Regression accuracy: 0.735


**3.2b — BERT + MLP**

In [35]:
from sklearn.neural_network import MLPClassifier

In [36]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# BERT values have a different scale from word counts, so we standardise them first.
bert_mlp = make_pipeline(StandardScaler(),
                         MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42))
bert_mlp.fit(X_train_bert, y_train)
acc = accuracy_score(y_test, bert_mlp.predict(X_test_bert))
results["BERT + MLP"] = acc
print(f"BERT + MLP accuracy: {acc:.3f}")

BERT + MLP accuracy: 0.728


## Part 4 — A Small Local Language Model (TinyLlama)

Instead of training a classifier, we ask a language model to do the task by reading an instruction. This is **prompting**. We will try several prompting styles on the same reviews and compare them.

**TinyLlama** is a 1.1-billion-parameter chat model that runs on your laptop with no API key. It is small, so its answers are noisier than a large model's, and the wording of the prompt matters a lot. The first run downloads about 2.2 GB.

In [37]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
llm_tok = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=torch.float32).eval()
print("TinyLlama loaded.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:02<00:00, 78.15it/s]

TinyLlama loaded.


In [38]:
@torch.no_grad()
def ask(prompt, max_new_tokens=5, temperature=0.0):
    """Send one message to TinyLlama and return its reply as text.

    temperature = 0   -> always pick the most likely next word (repeatable)
    temperature > 0   -> sample words at random, so replies vary between runs
    """
    messages = [{"role": "user", "content": prompt}]
    text = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tok(text, return_tensors="pt")
    gen_kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=llm_tok.eos_token_id)
    if temperature > 0:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen_kwargs.update(do_sample=False)
    out = llm.generate(**inputs, **gen_kwargs)
    return llm_tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [39]:
print(ask("In two sentences, what is a word embedding?", max_new_tokens=80))

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A word embedding is a mathematical representation of a word or a set of words, which is used to represent the meaning of the words in a text or a sentence. It is a vector-based representation that captures the context and relationships between words in a sentence.


### 4.1 Temperature

Temperature controls how random the model's word choices are. At temperature 0 the model always picks its top choice, so the answer is the same every time. Higher temperatures give more varied, and eventually less coherent, text.

In [40]:
prompt = "Describe a sunset in one sentence."
for temp in [0.0, 0.7, 1.2]:
    print(f"Temperature {temp}: {ask(prompt, max_new_tokens=40, temperature=temp)}\n")

[transformers] Both `max_new_tokens` (=40) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=40) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Temperature 0.0: A sunset is a breathtaking sight that captures the beauty of the sun as it sets over the horizon, creating a magical and awe-inspiring display of colors and



[transformers] Both `max_new_tokens` (=40) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Temperature 0.7: A sunset is a stunning sight of light, color, and atmosphere, as the sun slowly descends over the horizon, painting the sky in vibrant hues and creating a magical

Temperature 1.2: Sunset is the golden moment of the day when the sky turns into a deep and ethereal orange, blue, or pink. It's a breathtaking sight that aw



### 4.2 Zero-shot, few-shot, and persona prompts (sentiment)

We now classify 20 random test reviews with four prompt styles:

- **Zero-shot:** the task description only.
- **Few-shot:** the task description plus three worked examples. The examples are written by us and are *not* from the test set.
- **Persona:** the model is told to act as a film critic and to focus on the verdict rather than the plot summary.

For the classification output we read the reply: if it contains "positive" we predict positive, otherwise negative.

In [41]:
ZERO_SHOT = """Is the following movie review positive or negative? Answer with one word.

Review: {review}"""

FEW_SHOT = """Classify the sentiment of each movie review as Positive or Negative. Answer with one word.

Review: a smart, funny, beautifully shot film. one of the best of the year.
Sentiment: Positive

Review: the acting was wooden and the plot made no sense. i wanted my two hours back.
Sentiment: Negative

Review: it drags in the middle, but the final act is so moving that it earns every minute.
Sentiment: Positive

Review: {review}
Sentiment:"""

CUSTOM = """You are an experienced film critic. Ignore the plot summary. Focus on the reviewer's own opinion of the film, especially the final paragraph, where reviewers usually give their verdict.

Review: {review}

Does the reviewer recommend the film? Answer with one word: Positive or Negative."""

# The same 20 random test reviews are used for every method.
rng = np.random.default_rng(42)
subset = rng.choice(len(raw_test), size=20, replace=False)
y_subset = [y_test[i] for i in subset]

llm_results = {}   # prompt style -> accuracy on the 20 reviews

In [42]:
def classify(template, review, temperature=0.0):
    """Fill the prompt template with the review and return 'pos' or 'neg'."""
    reply = ask(template.format(review=review[:1200]), max_new_tokens=5, temperature=temperature)
    return "pos" if "positive" in reply.lower() else "neg"

In [43]:
def evaluate(template, name):
    """Classify the 20 reviews with one prompt, print accuracy and speed, and store the result."""
    t0 = time.time()
    preds = [classify(template, raw_test[i]) for i in subset]
    acc = accuracy_score(y_subset, preds)
    llm_results[name] = acc
    print(f"{name:<26} accuracy = {acc:.2f}   ({(time.time() - t0) / len(subset):.1f} s per review)")
    return preds

zero_preds = evaluate(ZERO_SHOT, "Zero-shot")
few_preds = evaluate(FEW_SHOT, "Few-shot")
persona_preds = evaluate(CUSTOM, "Persona (film critic)")

[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Zero-shot                  accuracy = 0.45   (4.5 s per review)


[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Few-shot                   accuracy = 0.40   (7.5 s per review)


[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Persona (film critic)      accuracy = 0.30   (7.9 s per review)


### 4.3 Chain-of-thought

Asking the model to reason step by step sometimes improves answers on harder reviews, because the reasoning is written out before the verdict. Small models produce reasoning that looks convincing, so always check whether the final verdict matches the steps above it.

In [44]:
COT = """Analyze the sentiment of this movie review step by step.
1. List the positive points the reviewer makes.
2. List the negative points.
3. Decide which side the reviewer ultimately favours.
4. On the last line, write the final verdict: Positive or Negative.

Review: {review}"""

mixed_reviews = [
    "The food was delicious but the service was terrible.",
    "I guess it wasn't completely awful.",
]
for review in mixed_reviews:
    print("=" * 70)
    print("REVIEW:", review)
    print("-" * 70)
    print(ask(COT.format(review=review), max_new_tokens=200))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


REVIEW: The food was delicious but the service was terrible.
----------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1. Positive points:
- The food was delicious, with a variety of options to choose from.
- The service was friendly and attentive, with waiters and waitresses coming to our table frequently to check on us and answer any questions we had.

2. Negative points:
- The waiters and waitresses were not very attentive, often leaving us waiting for our food for a long time.
- The service was inconsistent, with some times being great and others not so great.

3. Decision:
- The reviewer ultimately favours the positive points, as the food was delicious and the service was friendly and attentive.

4. Final verdict:
- Positive: Delicious food, friendly service
- Negative: Inconsistent service, long wait times
- Final verdict: Positive (3/5)
REVIEW: I guess it wasn't completely awful.
----------------------------------------------------------------------
1. Positive points:
- The cinematography was beautiful and immersive.
- The soundtrack was excellent.
- The performances were good, with some stando

### 4.4 Other NLP tasks by prompting

The same model can also do tasks we did not train it for: named-entity recognition, translation, and question answering from a passage. Each task is just a different instruction. These examples use sentences written for the demo, not reviews.

In [45]:
NER = """Extract the named entities from the text below.
Output one per line in the format:  TYPE: entity
Use only these types: PERSON, ORGANIZATION, LOCATION, DATE, MONEY. If there are none, output NONE.

Text: {text}"""

ner_text = "Apple CEO Tim Cook announced a $3 billion investment in Austin, Texas on January 15, 2024."
print("NER:")
print(ask(NER.format(text=ner_text), max_new_tokens=100))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


NER:
Type: entity

In: Apple CEO Tim Cook announced a $3 billion investment in Austin, Texas on January 15, 2024.


In [46]:
TRANSLATE = "Translate the following sentence into {language}. Reply with the translation only.\n\nSentence: {text}"

print(ask(TRANSLATE.format(language="French", text="The quick brown fox jumps over the lazy dog."),
          max_new_tokens=60))

[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


La chien lâche le chat s'éloigne le vif.


In [47]:
QA = """Answer the question using only the context. If the answer is not in the context, say "Not stated".

Context: {context}

Question: {question}"""

context = "Paris is the capital and largest city of France. It is situated on the River Seine in northern France."
for q in ["What is the capital of France?", "What river flows through Paris?", "What is the population of Paris?"]:
    print(f"Q: {q}")
    print(f"A: {ask(QA.format(context=context, question=q), max_new_tokens=30)}\n")

[transformers] Both `max_new_tokens` (=30) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the capital of France?


[transformers] Both `max_new_tokens` (=30) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: The capital of France is Paris.

Q: What river flows through Paris?


[transformers] Both `max_new_tokens` (=30) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Answer: The River Seine flows through Paris.

Q: What is the population of Paris?
A: The population of Paris is not stated in the given context.



### 4.5 Compare with the classical model

The classical SVM from Part 2 is scored on the same 20 reviews, so the comparison is fair.

In [48]:
svm_subset_pred = svm_bi.predict(tfidf_bi.transform([X_test[i] for i in subset]))
llm_results["TF-IDF uni+bigrams + SVM"] = accuracy_score(y_subset, svm_subset_pred)

comparison = pd.Series(llm_results, name="Accuracy on 20 reviews").sort_values(ascending=False).round(2)
display(comparison.to_frame())

,Accuracy on 20 reviews
TF-IDF uni+bigrams + SVM,0.85
Zero-shot,0.45
Few-shot,0.40
Persona (film critic),0.30


## Summary

The table below lists test accuracy on all 400 test reviews for Parts 2 and 3. TinyLlama was tested on 20 reviews only, so its result is shown separately above.

In [49]:
import pandas as pd

summary = pd.DataFrame({"Test accuracy (400 reviews)": results}).sort_values(
    "Test accuracy (400 reviews)", ascending=False).round(3)
display(summary)

,Test accuracy (400 reviews)
TF-IDF uni+bigrams + SVM,0.858
TF-IDF unigrams + SVM,0.852
TF-IDF + Logistic Regression,0.820
BoW + Naive Bayes,0.815
Word2Vec + Logistic Regression,0.798
Word2Vec + MLP,0.772
BERT + Logistic Regression,0.735
BERT + MLP,0.728


---
## Before you submit

1. **Restart** the kernel, then **Run All** from the toolbar. Every cell should run without errors.
2. Check that every cell with `# YOUR CODE HERE` has working code and no `raise NotImplementedError` is left.
3. Save the notebook with its outputs visible.